Hence, we get best results with TFIDF, tri-gram, 10000, LightGBM(earlier Adasyn has given better result but with these setting LightGBM is giving the best accruacy), learning_rate, n_estimators(increased from 50 to 200 to 50 to 500 thus increase accuracy) and max_depth(increased max_depth from 3 to 10 to 3 to 20 thus helps in increasing the accuracy).

We find out that instead of using Adasyn, we used LightBGM's built in class_weight parameter has giving good result. Optuna trails increased and increase the search space of the hyperparameters. thus accuracy has increased to 86% due to all.

In [1]:
from google.colab import userdata
import os

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = userdata.get('AWS_DEFAULT_REGION')
os.environ["satya_mlflow_ec2_uri"] = userdata.get('satya_mlflow_ec2_uri')

In [ ]:
! pip install lightgbm optuna
! aws sts get-caller-identity

Found existing installation: lightgbm 4.6.0
Uninstalling lightgbm-4.6.0:
  Successfully uninstalled lightgbm-4.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 12.7 MB/s eta 0:00:00
/bin/bash: line 1: aws: command not found


In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb
import pandas as pd
import optuna

In [11]:
from google.colab import drive
drive.mount('/content/drive')

# Drop rows with NaN values in 'clean_comment'
cleaned_dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Youtube_Comment_Sentiment_Analysis/reddit_preprocessing.csv').dropna()
cleaned_dataset.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(36662, 2)

In [12]:
# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

In [13]:
# Split the cleaned data into train and test sets (80-20 split)
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

In [14]:
# Apply TfidfVectorizer with trigram setting and max_features=10000
tfidf_cleaned = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)

In [15]:
# Fit the vectorizer on both train and test sets
X_train_tfidf_cleaned = tfidf_cleaned.fit_transform(X_train_cleaned)
X_test_tfidf_cleaned = tfidf_cleaned.transform(X_test_cleaned)

In [16]:
# Function to optimize LightGBM hyperparameters
def objective(trial):
    # Define hyperparameters to be tuned
    param = {
        "objective": "multiclass",
        "num_class": 3,  # Assuming 3 categories (-1, 0, 1)
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1), # 0.001 to 0.1
        "n_estimators": trial.suggest_int("n_estimators", 50, 500), # increased
        "max_depth": trial.suggest_int("max_depth", 3, 20), # increased
        "metric": "multi_logloss",
        "is_unbalance": True, # telling the LightGBM that the data we have is imbalanced
        "class_weight": "balanced" # adding this has increased accuracy
    }

    # Define the LightGBM model with the trial parameters
    model = lgb.LGBMClassifier(**param)

    # Perform cross-validation
    scores = cross_val_score(model, X_train_tfidf_cleaned, y_train_cleaned, cv=3, scoring='accuracy')

    # Return the average score across folds (accuracy)
    return scores.mean()

In [ ]:
# Create an Optuna study to optimize the hyperparameters
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) # increasing n_trails from 30 to 50 helped in increasing accuracy.

In [ ]:
# Extract the best hyperparameters
best_params = study.best_params
best_params

#### Now creating model with the best parameters we have learning rate, n_estimators and max_depth.

In [19]:
# Implementing the result of the above into this
best_model = lgb.LGBMClassifier(

    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance= True, # this is what we have added to get high accuracy
    class_weight= "balanced",
    reg_alpha= 0.1,  # L1 regularization : to prevent from overfitting
    reg_lambda= 0.1,  # L2 regularization
    learning_rate= 0.08,
    max_depth= 20,
    n_estimators=367
)

In [ ]:
# Fit the model on the resampled training data
best_model.fit(X_train_tfidf_cleaned, y_train_cleaned)

In [21]:
# Predict on the train set
y_train_pred = best_model.predict(X_train_tfidf_cleaned)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [22]:
# Calculate accuracy on the train set
accuracy_train = accuracy_score(y_train_cleaned, y_train_pred)
accuracy_train
# accuracy of 92 % is good let see for test dataset

0.9271369634150499

In [23]:
# Generate classification report of training data
report_train = classification_report(y_train_cleaned, y_train_pred)
print(report_train)
# Now, we can see that f1 score is good for all the three categories and

              precision    recall  f1-score   support

          -1       0.91      0.90      0.91      6601
           0       0.88      0.98      0.93     10134
           1       0.98      0.90      0.94     12594

    accuracy                           0.93     29329
   macro avg       0.92      0.93      0.92     29329
weighted avg       0.93      0.93      0.93     29329



In [24]:
# Predict on the test set
y_pred = best_model.predict(X_test_tfidf_cleaned)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
# Calculate accuracy on the test set
accuracy = accuracy_score(y_test_cleaned, y_pred)
accuracy
# accuracy on test data set is around 86 % that is good number not bad.

0.8633574253375154

In [26]:
# Generate classification report of test data
report = classification_report(y_test_cleaned, y_pred)
print(report)
# recall of -1 has improved from 0 to 78

              precision    recall  f1-score   support

          -1       0.80      0.78      0.79      1647
           0       0.84      0.97      0.90      2510
           1       0.92      0.82      0.87      3176

    accuracy                           0.86      7333
   macro avg       0.85      0.86      0.85      7333
weighted avg       0.87      0.86      0.86      7333



### To check model is working fine or not on some examples

In [27]:
import re
import numpy as np

# Assuming you have pre-trained tfidf_vectorizer and lgbm_model loaded
# tfidf_vectorizer: Your trained TF-IDF vectorizer
# lgbm_model: Your trained LightGBM model

# Function to clean and preprocess a YouTube comment (same as used during training)
def preprocess_comment(comment):
    # Lowercasing
    comment = comment.lower()

    # Remove special characters, URLs, punctuation, and extra spaces
    comment = re.sub(r"http\S+|www\S+|https\S+", '', comment, flags=re.MULTILINE)  # Remove URLs
    comment = re.sub(r'\W', ' ', comment)  # Remove special characters
    comment = re.sub(r'\s+', ' ', comment).strip()  # Remove extra spaces and newlines

    return comment

# Prediction function
def predict_sentiment(comment, tfidf_vectorizer, lgbm_model):
    # Step 1: Preprocess the YouTube comment
    cleaned_comment = preprocess_comment(comment)

    # Step 2: Transform the comment using the trained TF-IDF vectorizer
    comment_tfidf = tfidf_vectorizer.transform([cleaned_comment])

    # Step 3: Use the trained LightGBM model to predict the sentiment
    prediction = lgbm_model.predict(comment_tfidf)
    prediction_proba = lgbm_model.predict_proba(comment_tfidf)

    # Step 4: Get the predicted sentiment (label) and probability
    sentiment_class = np.argmax(prediction_proba)
    sentiment_proba = np.max(prediction_proba)

    # Step 5: Return the sentiment label and confidence
    return {
        'sentiment_class': int(prediction[0]),  # -1, 0, or 1 depending on your labels
        'confidence': sentiment_proba # as lightgbm also give probability of the prediction
    }

# Example usage:
test_comments_list = [ "I absolutely hate this video!",
"The explanations were confusing and the video quality was poor.",
"I didn’t learn anything useful. Really disappointed.",
"Wow, the explanation was so clear and helpful. Definitely subscribing!",
"This is the worst video I’ve seen on this topic, very misleading",
"Not much to say about this, just a standard video.",
"The video is okay, but I expected more depth in the content.",
"Superb content! Mazaa aa gaya dekh ke. Best video on this topic!",
"Poor video quality aur explanation bhi weak tha.",
"Yeh video theek tha, but I was expecting more depth."]
for test_comments in test_comments_list:
  result = predict_sentiment(test_comments, tfidf_cleaned, best_model)
  print(f"Predicted Sentiment: {result['sentiment_class']}, Confidence: {result['confidence']}")

Predicted Sentiment: 0, Confidence: 0.7813419986979802


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


This is our final model and we will develop first version of product using this.

Further Improvements:

1.   Word2Vec
2.   custom features
3.   Stacking
4.   Deep Learning (BERT)